# M2 Transformer Training Notebook (Kaggle-safe, token-level)

- Bám sát logic `models/m2_transformer/train.py`
- Thêm preprocessing bằng `pyvi` trước khi tạo vocab/train
- Hỗ trợ Kaggle + local, output ưu tiên `/kaggle/working` khi chạy trên Kaggle


In [ ]:
import os, json, csv, math, re, sys
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

IS_KAGGLE = Path('/kaggle').exists()
if IS_KAGGLE:
    get_ipython().run_line_magic('pip', 'install -q pyvi')

from pyvi import ViTokenizer

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config

KAGGLE_INPUT_CSV = Path('/kaggle/input/vietnamese-elementary-math-problems-with-images/data_warehouse.csv')
OUTPUT_BASE_DIR = Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT
M2_OUTPUT_DIR = OUTPUT_BASE_DIR / 'models' / 'm2_transformer'
M2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'IS_KAGGLE: {IS_KAGGLE}')
print(f'OUTPUT_DIR: {M2_OUTPUT_DIR}')


In [ ]:
def normalize_text(text):
    if text is None:
        return ''
    return re.sub(r'\\s+', ' ', str(text)).strip()

def word_tokenize_vi(text):
    clean = normalize_text(text)
    if not clean:
        return []
    return [tok for tok in ViTokenizer.tokenize(clean).split() if tok]

def preprocess_math_text(text):
    tokens = word_tokenize_vi(text)
    normalized = []
    for tok in tokens:
        t = re.sub(r'([=+\\-*/×÷()])', r' \\1 ', tok.strip())
        normalized.extend([x for x in t.split() if x])
    return normalized

def parse_list_cell(raw_value):
    if not raw_value or raw_value.strip() in ['', '[]', '[""]']:
        return []
    try:
        return json.loads(raw_value)
    except Exception:
        return []

def load_data_from_csv(csv_path):
    data_list = []
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            q = row.get('question', '').strip()
            a = row.get('answer', '').strip()
            images_path = parse_list_cell(row.get('images_path', ''))
            data_list.append((q, a, images_path))
    return data_list


In [ ]:
class CustomMathDataset(Dataset):
    def __init__(self, data_list, vocab, seq_len=128):
        self.data = data_list
        self.vocab = vocab
        self.seq_len = seq_len
        self.pad = vocab.get('<PAD>', 0)
        self.unk = vocab.get('<UNK>', 1)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        q, a, _ = self.data[idx]
        text = f'Câu hỏi: {q} Giải: {a}' if q and a else '<PAD>'
        toks = preprocess_math_text(text)
        ids = [self.vocab.get(t, self.unk) for t in toks]
        if len(ids) > self.seq_len:
            ids = ids[:self.seq_len]
        else:
            ids += [self.pad] * (self.seq_len - len(ids))
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return x, y

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerDecoderModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, dim_feedforward=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward, batch_first=True)
        self.decoder = nn.TransformerDecoder(layer, num_layers)
        self.fc = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        return mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))

    def forward(self, x):
        seq_len = x.size(1)
        tgt_mask = self.generate_square_subsequent_mask(seq_len).to(x.device)
        emb = self.embedding(x) * math.sqrt(self.d_model)
        emb = self.pos_encoder(emb)
        out = self.decoder(emb, emb, tgt_mask=tgt_mask)
        return self.fc(out)

def build_vocab(data_list):
    vocab = {'<PAD>': 0, '<UNK>': 1}
    idx = 2
    for q, a, _ in data_list:
        if not q or not a:
            continue
        for tok in preprocess_math_text(f'Câu hỏi: {q} Giải: {a}'):
            if tok not in vocab:
                vocab[tok] = idx
                idx += 1
    return vocab


In [ ]:
csv_path = KAGGLE_INPUT_CSV if IS_KAGGLE and KAGGLE_INPUT_CSV.exists() else (PROJECT_ROOT / 'data_warehouse.csv')
print(f'Loading data from: {csv_path}')
if not csv_path.exists():
    raise FileNotFoundError(f'Not found: {csv_path}')

data_list = load_data_from_csv(csv_path)
train_size = int(len(data_list) * 0.8)
train_data = data_list[:train_size]
test_data = data_list[train_size:]

vocab = build_vocab(train_data)
vocab_size = len(vocab)

vocab_path = M2_OUTPUT_DIR / 'vocab.json'
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False)

meta_path = M2_OUTPUT_DIR / 'preprocess_meta.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump({'tokenizer':'pyvi.ViTokenizer','level':'word','seq_len':128,'special_tokens':['<PAD>','<UNK>']}, f, ensure_ascii=False, indent=2)

print(f'Train: {len(train_data)} | Test: {len(test_data)} | Vocab: {vocab_size}')
print(f'Saved vocab: {vocab_path}')


In [ ]:
train_dataset = CustomMathDataset(train_data, vocab, seq_len=128)
test_dataset = CustomMathDataset(test_data, vocab, seq_len=128)

train_loader = DataLoader(train_dataset, batch_size=config.TRAIN_BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=config.TRAIN_BATCH_SIZE, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TransformerDecoderModel(vocab_size).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=vocab.get('<PAD>', 0))
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 5

csv_log_path = M2_OUTPUT_DIR / 'logs' / 'training_log.csv'
csv_log_path.parent.mkdir(parents=True, exist_ok=True)
with open(csv_log_path, 'w', newline='') as f:
    csv.writer(f).writerow(['step', 'epoch', 'loss', 'learning_rate'])

global_step = 0
train_losses = []
model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out.view(-1, vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()
        global_step += 1
        epoch_loss += loss.item()
        if global_step % 10 == 0:
            with open(csv_log_path, 'a', newline='') as f:
                csv.writer(f).writerow([global_step, epoch, round(loss.item(), 4), optimizer.param_groups[0]['lr']])
    avg = epoch_loss / max(1, len(train_loader))
    train_losses.append(avg)
    print(f'Epoch {epoch+1}/{num_epochs} - avg loss: {avg:.4f}')

print('Training finished')


In [ ]:
model.eval()
test_loss = 0.0
batches = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out.view(-1, vocab_size), y.view(-1))
        test_loss += loss.item()
        batches += 1
avg_test_loss = test_loss / max(1, batches)
print(f'Average Test Loss: {avg_test_loss:.4f}')


In [ ]:
final_dir = M2_OUTPUT_DIR / 'final'
final_dir.mkdir(parents=True, exist_ok=True)
model_path = final_dir / 'model.pt'
torch.save(model.state_dict(), model_path)
print(f'Saved model: {model_path}')

plt.figure(figsize=(10,5))
plt.plot(train_losses, marker='o', label='Train Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('M2 Transformer Training Loss')
plt.grid(True, alpha=0.3)
plt.legend()
plot_path = (Path('/kaggle/working') / 'm2_training_loss.png') if IS_KAGGLE else (PROJECT_ROOT / 'models/m2_transformer/training_loss.png')
plot_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(plot_path, dpi=100)
plt.show()
print(f'Saved plot: {plot_path}')
